# 🎓 Przykładowy Projekt Zaliczeniowy
## Analiza i Prognozowanie Opóźnień Lotów ✈️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/example/projekt-zaliczeniowy-przyklad.ipynb)

---

| | |
|---|---|
| **Autor** | Jan Kowalski |
| **Nr indeksu** | 123456 |
| **Kurs** | Python w Zastosowaniach Big Data |
| **Semestr** | Letni 2025/2026 |
| **Data** | Czerwiec 2026 |

---

## 🎯 Cel projektu

Celem projektu jest zbadanie **czynników wpływających na opóźnienia lotów** oraz budowa modelu predykcyjnego, który — na podstawie charakterystyki lotu — przewidzi, czy rejs będzie opóźniony o **więcej niż 15 minut**.

**Pytanie badawcze:**
> *Które cechy lotu (linia lotnicza, godzina odlotu, dzień tygodnia, warunki pogodowe) mają największy wpływ na prawdopodobieństwo opóźnienia?*

## 🛠️ Technologie

| Technologia | Zastosowanie |
|---|---|
| **Apache Spark (PySpark)** | Przetwarzanie i analiza dużych zbiorów danych |
| **Spark SQL** | Zaawansowane zapytania i agregacje |
| **MLlib (Random Forest)** | Klasyfikacja — prognozowanie opóźnień |
| **Matplotlib / Seaborn** | Wizualizacje i wykresy |

## 📋 Spis treści

1. Konfiguracja środowiska
2. Generowanie i ładowanie danych
3. Eksploracyjna Analiza Danych (EDA)
4. Analiza z Spark SQL
5. Feature Engineering
6. Modelowanie ML (Random Forest)
7. Ewaluacja modelu
8. Wnioski i rekomendacje

---

> 💡 **Uwaga dla studentów:** Ten notebook to **przykładowy** projekt zaliczeniowy — ilustruje oczekiwaną strukturę, poziom szczegółowości i sposób prezentacji wyników. Twój projekt powinien dotyczyć **innego tematu** i **własnego zbioru danych**!

---
## ⚠️ Ważna informacja: To jest przykładowy projekt!

> **Ten notebook demonstruje JEDEN możliwy sposób realizacji projektu.**
> Twój projekt **musi** dotyczyć innego tematu i korzystać z własnego zbioru danych.
> Poniżej znajdziesz inspiracje, jak ten sam cel można osiągnąć innymi technologiami z kursu.

---

## 🧭 Alternatywne technologie — możesz użyć dowolnej z niż wymienionych!

Poniżej znajdziesz mini-przykłady pokazujące, jak **te same zadania** (EDA, przetwarzanie, ML)
można wykonać innymi narzędziami omawianymi na kursie. Wybierz to, co najlepiej pasuje do Twojego tematu.

---

### 1️⃣ Google BigQuery (GCP) — SQL w chmurze na petabajtach

**Kiedy używać?** Gdy Twój zbior danych jest w Google Cloud lub korzystasz z BigQuery Public Datasets.

```python
from google.cloud import bigquery

client = bigquery.Client(project="twój-projekt-gcp")

# Analiza danych lotniczych bezpośrednio w BigQuery (petabajty danych!)
query = """
    SELECT
        carrier,
        ROUND(AVG(dep_delay), 2) AS avg_delay,
        COUNTIF(dep_delay > 15) / COUNT(*) * 100 AS pct_delayed
    FROM `bigquery-public-data.airline_ontime_data.flights`
    WHERE year = 2023
    GROUP BY carrier
    ORDER BY pct_delayed DESC
"""

df = client.query(query).to_dataframe()
print(df.head(10))
```

---

### 2️⃣ Spark Streaming — analiza danych w czasie rzeczywistym

**Kiedy używać?** Gdy dane spływają na żywo (np. logi z lotniska, dane z IoT, transakcje).

```python
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType

spark = SparkSession.builder.appName("FlightStreamingDemo").getOrCreate()

# Schema danych przychodzących (np. z Kafki lub gniazda sieciowego)
schema = StructType() \
    .add("carrier", StringType()) \
    .add("dep_delay", DoubleType()) \
    .add("origin", StringType())

# Źródło strumieniowe (socket — do celów demonstracyjnych)
stream_df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Agregacja w oknie czasowym 1 minuty
windowed = (
    stream_df
    .withColumn("timestamp", F.current_timestamp())
    .groupBy(F.window("timestamp", "1 minute"), "carrier")
    .agg(
        F.avg("dep_delay").alias("avg_delay_1min"),
        F.count("*").alias("flights_count")
    )
)

# Uruchomienie zapytania strumieniowego
query = windowed.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination(60)  # Czeka 60 sekund, potem kończy
```

---

### 3️⃣ scikit-learn — klasyczne ML na mniejszych zbiorach

**Kiedy używać?** Gdy zbior danych mieści się w pamięci RAM (do kilku GB) — scikit-learn jest szybsze i wygodniejsze niż MLlib.

```python
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

# Przygotowanie danych
X = pdf[["dep_hour", "month", "day_of_week", "distance"]].copy()
for col in ["carrier", "origin"]:
    X[col] = LabelEncoder().fit_transform(pdf[col])
y = pdf["is_delayed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Grid Search z Cross-Validation (scikit-learn API)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth":    [5, 10, None],
    "min_samples_leaf": [1, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,               # 5-fold cross-validation
    scoring="roc_auc",
    n_jobs=-1,          # wszystkie rdzenie CPU
    verbose=1,
)
grid_search.fit(X_train, y_train)

print(f"Najlepsze parametry: {grid_search.best_params_}")
print(f"CV AUC-ROC:         {grid_search.best_score_:.4f}")

y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Punktualny", "Opozniony"]))
```

---

### 4️⃣ MLlib — inne algorytmy klasyfikacji

**Kiedy używać?** Gdy zostajemy w środowisku Spark, ale chcemy porównać różne algorytmy.

```python
from pyspark.ml.classification import (
    GBTClassifier,            # Gradient Boosted Trees — często najlepsza jakość
    LogisticRegression,       # Szybki baseline, interpretowalny
    LinearSVC,                # Support Vector Machine
    DecisionTreeClassifier,   # Pojedyncze drzewo — bardzo interpretowalny
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Porównanie kilku algorytmów na tych samych danych
classifiers = {
    "Logistic Regression": LogisticRegression(labelCol="is_delayed", maxIter=20),
    "Random Forest":       RandomForestClassifier(labelCol="is_delayed", numTrees=50),
    "Gradient Boosted Trees": GBTClassifier(labelCol="is_delayed", maxIter=20),
}

evaluator = BinaryClassificationEvaluator(labelCol="is_delayed")
results = {}

for name, clf in classifiers.items():
    # Każdy classifier wymaga pipeline z preprocessing (StringIndexer, VectorAssembler)
    pipeline_clf = Pipeline(stages=indexers + [assembler, clf])
    fitted = pipeline_clf.fit(train_df)
    preds  = fitted.transform(test_df)
    auc    = evaluator.evaluate(preds)
    results[name] = auc
    print(f"  {name:<25} AUC-ROC = {auc:.4f}")

best_clf = max(results, key=results.get)
print(f"\nNajlepszy algorytm: {best_clf} (AUC = {results[best_clf]:.4f})")
```

---

### 5️⃣ Apache Kafka + PySpark — ingestion + przetwarzanie

**Kiedy używać?** Gdy chcesz zaprezentować kompletny potok Big Data: dane płyną z Kafki, Spark je przetwarza.

```python
# Odczyt strumienia z topiku Kafka
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "flight-events")          # nazwa topiku
    .option("startingOffsets", "latest")
    .load()
)

# Wiadomości Kafki są bajtami — deserializujemy JSON
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

flight_schema = StructType() \
    .add("carrier", StringType()) \
    .add("dep_delay", DoubleType()) \
    .add("origin", StringType())

parsed = kafka_df.select(
    from_json(col("value").cast("string"), flight_schema).alias("data")
).select("data.*")

# Statystyki na żywo: średnie opóźnienie per linia
live_stats = parsed.groupBy("carrier").avg("dep_delay")

live_stats.writeStream \
    .outputMode("complete") \
    .format("console") \
    .trigger(processingTime="10 seconds") \
    .start() \
    .awaitTermination()
```

---

> 📚 **Pamiętaj:** Powyższe przykłady to **punkty startowe**, nie gotowe projekty.
> W swoim projekcie pokaż, że **rozumiesz** wybrany stos technologiczny —
> pocałej linijki kodu powinieneś umieć wyjaśnić prowadzącemu na obronie! 💪


---
## 1. Konfiguracja środowiska 🔧

Instalujemy i importujemy wszystkie niezbędne biblioteki. Na początku każdego projektu warto zebrać wszystkie importy w jednym miejscu — ułatwia to debugowanie i późniejsze uruchamianie.

In [ ]:
# @title 🔧 Instalacja i importy
!pip install pyspark -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Ustawienia wizualizacji
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.dpi"] = 110

print("✅ Wszystkie biblioteki załadowane pomyślnie!")

In [ ]:
# @title ⚡ Inicjalizacja SparkSession
spark = (
    SparkSession.builder
    .appName("Analiza Opóźnień Lotów")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Apache Spark {spark.version} uruchomiony!")
print(f"🖥️  Tryb: {spark.sparkContext.master}")
print(f"🔢 Domyślna liczba partycji: {spark.conf.get('spark.sql.shuffle.partitions')}")

---
## 2. Generowanie i ładowanie danych 📊

W tym projekcie korzystamy z **syntetycznego zbioru danych** inspirowanego rzeczywistymi danymi US Bureau of Transportation Statistics (BTS). Zbiór zawiera **500 000 rekordów**, co czyni go odpowiednim kandydatem do przetwarzania Big Data.

**Opis kolumn:**

| Kolumna | Typ | Opis |
|---|---|---|
| `carrier` | String | Kod linii lotniczej (AA, DL, UA, ...) |
| `origin` | String | Kod lotniska wylotu |
| `month` | Int | Miesiąc lotu (1–12) |
| `day_of_week` | Int | Dzień tygodnia (1=Pon, 7=Nd) |
| `dep_hour` | Int | Godzina odlotu (5–23) |
| `distance` | Int | Odległość w milach |
| `weather_delay` | Double | Składowa opóźnienia: warunki pogodowe (min) |
| `carrier_delay` | Double | Składowa opóźnienia: linia lotnicza (min) |
| `nas_delay` | Double | Składowa opóźnienia: system kontroli ruchu (min) |
| `dep_delay` | Double | Łączne opóźnienie odlotu (min) |
| `is_delayed` | Int | **Zmienna docelowa**: 1 = opóźniony >15 min, 0 = punktualny |

In [ ]:
# @title 🏗️ Generowanie syntetycznych danych
np.random.seed(42)
N = 500_000  # pół miliona rekordów

carriers   = ['AA', 'DL', 'UA', 'WN', 'B6', 'AS']
carr_probs = [0.25, 0.20, 0.18, 0.17, 0.12, 0.08]
airports   = ['JFK', 'LAX', 'ORD', 'ATL', 'DFW', 'SFO', 'SEA', 'MIA', 'BOS', 'DEN']

carrier_arr     = np.random.choice(carriers, N, p=carr_probs)
origin_arr      = np.random.choice(airports, N)
month_arr       = np.random.randint(1, 13, N)
day_of_week_arr = np.random.randint(1, 8, N)
dep_hour_arr    = np.random.choice(range(5, 24), N)
distance_arr    = np.clip(np.random.lognormal(6.5, 0.7, N).astype(int), 100, 5000)

# Bias opóźnień per linia (im wyższy → więcej opóźnień)
carrier_bias_map = {'AA': 6.0, 'DL': 2.5, 'UA': 9.0, 'WN': 4.0, 'B6': 12.0, 'AS': 1.5}
carrier_bias     = np.array([carrier_bias_map[c] for c in carrier_arr])

# Składowe opóźnień z rozkładu wykładniczego
weather_delay = np.random.exponential(2.5, N)
carrier_delay = np.random.exponential(1.0, N) * carrier_bias
nas_delay     = np.random.exponential(2.0, N)

# Sezonowość: więcej opóźnień w zimie i lecie
season_factor = np.where(
    np.isin(month_arr, [12, 1, 6, 7, 8]), 1.4,
    np.where(np.isin(month_arr, [2, 3, 11]), 1.1, 0.9)
)

dep_delay = ((weather_delay + carrier_delay + nas_delay) * season_factor - 8).round(1)
dep_delay = np.clip(dep_delay, -30, 300)
is_delayed = (dep_delay > 15).astype(int)

# Tworzymy pandas DataFrame, a następnie konwertujemy do Spark
pdf = pd.DataFrame({
    'carrier':       carrier_arr,
    'origin':        origin_arr,
    'month':         month_arr.astype('int32'),
    'day_of_week':   day_of_week_arr.astype('int32'),
    'dep_hour':      dep_hour_arr.astype('int32'),
    'distance':      distance_arr.astype('int32'),
    'weather_delay': weather_delay.round(1),
    'carrier_delay': carrier_delay.round(1),
    'nas_delay':     nas_delay.round(1),
    'dep_delay':     dep_delay,
    'is_delayed':    is_delayed.astype('int32'),
})

# Konwersja do Apache Spark DataFrame
df = spark.createDataFrame(pdf)
df.cache()  # Cachujemy w pamięci, bo będziemy wielokrotnie odpytywać

print(f"✅ Wygenerowano {N:,} rekordów")
print(f"📊 Odsetek opóźnionych lotów: {is_delayed.mean()*100:.1f}%")
print(f"🔢 Liczba partycji: {df.rdd.getNumPartitions()}")

In [ ]:
# @title 🔍 Schemat i podgląd danych
print("=" * 50)
print("SCHEMAT DANYCH")
print("=" * 50)
df.printSchema()

print("\n" + "=" * 50)
print("PIERWSZE 5 REKORDÓW")
print("=" * 50)
df.show(5, truncate=False)

print(f"\n📐 Rozmiar zbioru: {df.count():,} wierszy × {len(df.columns)} kolumn")

---
## 3. Eksploracyjna Analiza Danych (EDA) 🔍

Zanim przejdziemy do modelowania, musimy dobrze **poznać nasze dane**. EDA odpowiada na pytania:
- Jak rozkładają się opóźnienia?
- Które linie lotnicze opóźniają się najczęściej?
- O której godzinie loty są najbardziej punktualne?
- Czy sezon roku ma wpływ na opóźnienia?

In [ ]:
# @title 📈 Statystyki opisowe
print("=" * 50)
print("STATYSTYKI OPISOWE — zmienne numeryczne")
print("=" * 50)
df.select('dep_delay', 'weather_delay', 'carrier_delay', 'nas_delay', 'distance').describe().show()

# Rozkład zmiennej docelowej
print("=" * 50)
print("ROZKŁAD ZMIENNEJ DOCELOWEJ (is_delayed)")
print("=" * 50)
df.groupBy('is_delayed').count().withColumn(
    'procent', F.round(F.col('count') / df.count() * 100, 2)
).orderBy('is_delayed').show()

In [ ]:
# @title ✈️ Analiza per linia lotnicza
carrier_stats = (
    df.groupBy('carrier')
    .agg(
        F.count('*').alias('liczba_lotow'),
        F.round(F.mean('dep_delay'), 2).alias('srednie_opoznienie'),
        F.round(F.mean('is_delayed') * 100, 1).alias('pct_opoznionych'),
        F.round(F.stddev('dep_delay'), 2).alias('odch_std_opoznienia'),
    )
    .orderBy(F.desc('pct_opoznionych'))
)

print("Statystyki opóźnień per linia lotnicza:")
carrier_stats.show()

# Konwertujemy do pandas na potrzeby wizualizacji
carrier_pdf = carrier_stats.toPandas()

In [ ]:
# @title 🕐 Analiza per godzina odlotu i miesiąc
hourly_stats = (
    df.groupBy('dep_hour')
    .agg(F.round(F.mean('is_delayed') * 100, 1).alias('pct_opoznionych'))
    .orderBy('dep_hour')
    .toPandas()
)

monthly_stats = (
    df.groupBy('month')
    .agg(F.round(F.mean('is_delayed') * 100, 1).alias('pct_opoznionych'))
    .orderBy('month')
    .toPandas()
)

print("Opóźnienia per godzina:")
print(hourly_stats.to_string(index=False))

In [ ]:
# @title 📊 Wizualizacje EDA
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analiza Opóźnień Lotów — Eksploracja Danych', fontsize=15, fontweight='bold', y=1.01)

# 1. % opóźnionych per linia lotnicza
ax1 = axes[0, 0]
bars = ax1.bar(
    carrier_pdf['carrier'],
    carrier_pdf['pct_opoznionych'],
    color=sns.color_palette('husl', len(carrier_pdf))
)
ax1.set_title('% Opóźnionych lotów per linia', fontweight='bold')
ax1.set_xlabel('Linia lotnicza')
ax1.set_ylabel('% opóźnionych (>15 min)')
for bar, val in zip(bars, carrier_pdf['pct_opoznionych']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

# 2. Opóźnienia wg godziny odlotu
ax2 = axes[0, 1]
ax2.plot(hourly_stats['dep_hour'], hourly_stats['pct_opoznionych'],
         marker='o', linewidth=2, color='steelblue', markersize=5)
ax2.fill_between(hourly_stats['dep_hour'], hourly_stats['pct_opoznionych'], alpha=0.2, color='steelblue')
ax2.set_title('% Opóźnionych wg godziny odlotu', fontweight='bold')
ax2.set_xlabel('Godzina odlotu')
ax2.set_ylabel('% opóźnionych')
ax2.set_xticks(range(5, 24))

# 3. Sezonowość — opóźnienia wg miesiąca
ax3 = axes[1, 0]
month_names = ['Sty', 'Lut', 'Mar', 'Kwi', 'Maj', 'Cze',
               'Lip', 'Sie', 'Wrz', 'Paź', 'Lis', 'Gru']
ax3.bar(monthly_stats['month'], monthly_stats['pct_opoznionych'],
        color=sns.color_palette('coolwarm', 12))
ax3.set_xticks(range(1, 13))
ax3.set_xticklabels(month_names, rotation=45)
ax3.set_title('Sezonowość opóźnień', fontweight='bold')
ax3.set_xlabel('Miesiąc')
ax3.set_ylabel('% opóźnionych')

# 4. Rozkład dep_delay (histogram)
ax4 = axes[1, 1]
delay_sample = pdf['dep_delay'].sample(5000, random_state=42)
ax4.hist(delay_sample[delay_sample <= 120], bins=50, color='coral', edgecolor='white', alpha=0.8)
ax4.axvline(15, color='red', linestyle='--', linewidth=2, label='Próg: 15 min')
ax4.set_title('Rozkład opóźnień dep_delay (próbka 5k)', fontweight='bold')
ax4.set_xlabel('Opóźnienie (minuty)')
ax4.set_ylabel('Liczba lotów')
ax4.legend()

plt.tight_layout()
plt.savefig('eda_visualizations.png', dpi=110, bbox_inches='tight')
plt.show()
print("📊 Wykresy zapisane do: eda_visualizations.png")

---
## 4. Analiza z Spark SQL 🗃️

Spark SQL umożliwia pisanie **standardowych zapytań SQL** bezpośrednio na Spark DataFrames — bez konieczności używania syntaksu PySpark. Jest szczególnie przydatny dla osób znających SQL oraz przy złożonych aggregacjach i operacjach okienkowych.

In [ ]:
# @title 🔖 Rejestracja widoku tymczasowego
df.createOrReplaceTempView("flights")
print("✅ Widok 'flights' zarejestrowany — można teraz pisać SQL!")

In [ ]:
# @title 🏆 SQL: Ranking linii lotniczych wg % opóźnień
query_ranking = """
SELECT
    carrier,
    COUNT(*) AS liczba_lotow,
    ROUND(AVG(dep_delay), 2) AS avg_opoznienie_min,
    ROUND(AVG(is_delayed) * 100, 1) AS pct_opoznionych,
    ROUND(MAX(dep_delay), 0) AS max_opoznienie_min
FROM flights
GROUP BY carrier
ORDER BY pct_opoznionych DESC
"""

print("🏆 Ranking linii lotniczych — % opóźnionych lotów:")
spark.sql(query_ranking).show()

In [ ]:
# @title 🪟 SQL: Funkcje okienkowe — ranking lotnisk per linia
query_window = """
WITH airport_carrier_stats AS (
    SELECT
        carrier,
        origin,
        ROUND(AVG(is_delayed) * 100, 1) AS pct_opoznionych,
        COUNT(*) AS liczba_lotow
    FROM flights
    GROUP BY carrier, origin
    HAVING COUNT(*) >= 500
)
SELECT
    carrier,
    origin,
    pct_opoznionych,
    liczba_lotow,
    RANK() OVER (PARTITION BY carrier ORDER BY pct_opoznionych DESC) AS rank_w_linii
FROM airport_carrier_stats
QUALIFY rank_w_linii <= 3
ORDER BY carrier, rank_w_linii
"""

# Uwaga: QUALIFY nie jest obsługiwane w starszych wersjach Spark SQL — alternatywa poniżej
query_window_alt = """
SELECT * FROM (
    SELECT
        carrier,
        origin,
        pct_opoznionych,
        liczba_lotow,
        RANK() OVER (PARTITION BY carrier ORDER BY pct_opoznionych DESC) AS rank_w_linii
    FROM (
        SELECT
            carrier, origin,
            ROUND(AVG(is_delayed) * 100, 1) AS pct_opoznionych,
            COUNT(*) AS liczba_lotow
        FROM flights
        GROUP BY carrier, origin
        HAVING COUNT(*) >= 500
    )
) WHERE rank_w_linii <= 3
ORDER BY carrier, rank_w_linii
"""

print("🪟 TOP 3 lotniska (najwyższy % opóźnień) per linia lotnicza:")
spark.sql(query_window_alt).show(30)

In [ ]:
# @title 📅 SQL: Analiza sezonowa i tygodniowa
query_season = """
SELECT
    CASE
        WHEN month IN (12, 1, 2)  THEN 'Zima ❄️'
        WHEN month IN (3, 4, 5)   THEN 'Wiosna 🌸'
        WHEN month IN (6, 7, 8)   THEN 'Lato ☀️'
        ELSE 'Jesień 🍂'
    END AS sezon,
    CASE day_of_week
        WHEN 1 THEN 'Pon' WHEN 2 THEN 'Wt' WHEN 3 THEN 'Śr'
        WHEN 4 THEN 'Czw' WHEN 5 THEN 'Pt' WHEN 6 THEN 'Sob'
        ELSE 'Nd'
    END AS dzien,
    ROUND(AVG(is_delayed) * 100, 1) AS pct_opoznionych,
    COUNT(*) AS liczba_lotow
FROM flights
GROUP BY sezon, dzien, month, day_of_week
ORDER BY month, day_of_week
LIMIT 28
"""

print("📅 Opóźnienia wg sezonu i dnia tygodnia:")
spark.sql(query_season).show(28)

---
## 5. Feature Engineering ⚙️

**Feature engineering** to proces tworzenia nowych, lepszych cech z istniejących danych. Dobry feature engineering często ma **większy wpływ na jakość modelu** niż wybór algorytmu!

Tworzymy:
- `is_weekend` — czy lot jest w weekend?
- `time_of_day` — pora dnia (ranek, południe, wieczór, noc)
- `sezon` — pora roku
- `total_delay_factors` — łączna "presja" opóźnień (suma składowych)

In [ ]:
# @title ⚙️ Tworzenie nowych cech
df_feat = df.withColumns({
    # Weekend vs. dzień roboczy
    'is_weekend': F.when(F.col('day_of_week').isin([6, 7]), 1).otherwise(0),

    # Pora dnia
    'time_of_day': F.when(F.col('dep_hour').between(5, 11), 'ranek')
                    .when(F.col('dep_hour').between(12, 16), 'poludnie')
                    .when(F.col('dep_hour').between(17, 20), 'wieczor')
                    .otherwise('noc'),

    # Sezon
    'sezon': F.when(F.col('month').isin([12, 1, 2]), 'zima')
              .when(F.col('month').isin([3, 4, 5]), 'wiosna')
              .when(F.col('month').isin([6, 7, 8]), 'lato')
              .otherwise('jesien'),

    # Długość lotu: krótki / średni / długi
    'flight_length': F.when(F.col('distance') < 500, 'krotki')
                      .when(F.col('distance') < 1500, 'sredni')
                      .otherwise('dlugi'),

    # Łączna presja opóźnień
    'total_delay_factors': F.round(
        F.col('weather_delay') + F.col('carrier_delay') + F.col('nas_delay'), 2
    ),
})

print("✅ Nowe cechy dodane!")
df_feat.select(
    'carrier', 'dep_hour', 'time_of_day', 'is_weekend', 'sezon',
    'flight_length', 'total_delay_factors', 'is_delayed'
).show(8)

In [ ]:
# @title 🔗 Analiza korelacji cech numerycznych
num_cols = ['dep_hour', 'month', 'day_of_week', 'distance',
            'weather_delay', 'carrier_delay', 'nas_delay',
            'total_delay_factors', 'is_weekend', 'is_delayed']

# Pobieramy próbkę 10k do obliczenia macierzy korelacji
corr_pdf = df_feat.select(num_cols).sample(fraction=0.02, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = corr_pdf.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax, square=True,
            annot_kws={'size': 9})
ax.set_title('Macierz korelacji — cechy numeryczne', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=110, bbox_inches='tight')
plt.show()

# Korelacja z is_delayed
print("\n📊 Korelacja z 'is_delayed' (zmienna docelowa):")
print(corr_matrix['is_delayed'].drop('is_delayed').sort_values(ascending=False).to_string())

---
## 6. Modelowanie ML — Random Forest + Strojenie hiperparametrów 🌲

Budujemy model klasyfikacji binarnej, który przewiduje, czy lot będzie opóźniony (`is_delayed = 1`) czy punktualny (`is_delayed = 0`). Następnie **strojemy hiperparametry** (hyperparameter tuning) korzystając z `CrossValidator` i `ParamGridBuilder`.

### Dlaczego Random Forest?
- Działa dobrze z **mieszanymi typami cech** (numeryczne + kategoryczne)
- **Odporny na overfitting** — dzięki baggingowi i losowemu wyborowi cech na każdym węźle
- Daje interpretowalne **feature importance** — wiemy, które cechy naprawdę mają znaczenie
- Skaluje się dobrze w MLlib na klastrze Spark
- Nie wymaga normalizacji/skalowania cech (w przeciwieństwie do SVM, regresji logistycznej)

### Pipeline MLlib
Używamy `Pipeline` — sekwencji transformacji zakończonej estymatorem. To **kluczowa dobra praktyka**, ponieważ:
- Zapobiega **data leakage** — `StringIndexer` jest fitowany wyłącznie na zbiorze treningowym,
  a następnie aplikowany do testowego
- Ułatwia **deployment modelu** — zapisujemy jeden obiekt, który sam wykonuje preprocessing
- Kod jest czysty, modularny i łatwy do modyfikacji

### Strojenie hiperparametrów: Grid Search + k-fold Cross-Validation

**Hiperparametry** to ustawienia modelu, których algorytm sam nie uczy się z danych. Złe wartości → słabszy model!

**Grid Search** systematycznie sprawdza wszystkie kombinacje z siatki:

```
numTrees = [30, 80]   →  2 wartości
maxDepth = [6, 10]    →  2 wartości
─────────────────────────────
Razem: 2 × 2 = 4 kombinacje
```

**k-fold Cross-Validation** dzieli zbiór treningowy na k równych części (foldów). Dla każdej kombinacji parametrów przeprowadza k treninów:

```
k=3, fold 1: [VALID][TRAIN][TRAIN]  →  model₁, AUC₁
k=3, fold 2: [TRAIN][VALID][TRAIN]  →  model₂, AUC₂
k=3, fold 3: [TRAIN][TRAIN][VALID]  →  model₃, AUC₃
─────────────────────────────────────────────────────
Wynik dla kombinacji = mean(AUC₁, AUC₂, AUC₃)
```

Łącznie: **4 kombinacje × 3 foldy = 12 pełnych treninów**. Wybieramy kombinację z najwyższą średną AUC-ROC.

> ⚠️ **Uwaga:** Na dużym zbiorze Cross-Validation jest kosztowne obliczeniowo. Alternatywą jest `TrainValidationSplit` — jeden podział zamiast k-foldów (szybciej, ale mniej rzetelnie).


In [ ]:
# @title 🏗️ Budowa Pipeline ML
# Cechy kategoryczne do zaindeksowania
cat_cols = ['carrier', 'origin', 'time_of_day', 'sezon', 'flight_length']
# Cechy numeryczne
num_cols_ml = ['month', 'day_of_week', 'dep_hour', 'distance',
               'total_delay_factors', 'is_weekend']

# Krok 1: StringIndexer — koduje kategorie jako liczby
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid='keep')
    for col in cat_cols
]

# Krok 2: VectorAssembler — scala wszystkie cechy w jeden wektor
feature_cols = num_cols_ml + [f"{col}_idx" for col in cat_cols]
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features',
    handleInvalid='keep'
)

# Krok 3: Random Forest Classifier
rf = RandomForestClassifier(
    labelCol='is_delayed',
    featuresCol='features',
    numTrees=50,
    maxDepth=8,
    seed=42
)

# Składamy Pipeline
pipeline = Pipeline(stages=indexers + [assembler, rf])

print("✅ Pipeline skonfigurowany:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  Krok {i+1}: {type(stage).__name__}")

In [ ]:
# @title 🚂 Krok 1: Podział danych i model bazowy (bez strojenia)
# Podział: 80% trening, 20% test
# seed=42 zapewnia powtórzalność — za każdym uruchomieniem ten sam podział
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)

# Cachujemy podzbiory — CrossValidator będzie je wielokrotnie odczytywać
train_df.cache()
test_df.cache()

print(f"Zbior treningowy: {train_df.count():,} rekordow")
print(f"Zbior testowy:    {test_df.count():,} rekordow")

# Model bazowy — domyślne hiperparametry (numTrees=50, maxDepth=8)
print("\n Trenuję model bazowy (domyslne hiperparametry)...")
model_baseline = pipeline.fit(train_df)

print("Model bazowy wytrenowany!")

# Predykcje bazowe
predictions_baseline = model_baseline.transform(test_df)
predictions_baseline.select(
    "carrier", "dep_hour", "sezon", "is_delayed", "prediction",
    F.round(F.col("probability")[1], 3).alias("prob_opoznienia")
).show(8, truncate=False)


### 🔎 Krok 2: Grid Search z Cross-Validation

Teraz szukamy optymalnych hiperparametrów. Używamy:
- **`ParamGridBuilder`** — do zdefiniowania siatki wartości
- **`CrossValidator`** — do wykonania k-fold CV dla każdej kombinacji
- **`BinaryClassificationEvaluator`** z metryką AUC-ROC — do oceny modeli

| Hiperparametr | Co reguluje? | Testowane wartości |
|---|---|---|
| `numTrees` | Liczba drzew w lesie (więcej → stabilniej, wolniej) | 30, 80 |
| `maxDepth` | Głębokość drzewa (głębiej → bardziej złożony model, ryzyko overfittingu) | 6, 10 |
| `minInstancesPerNode` | Min. obserwacji w liściu (wyżej → regularyzacja) | 2 (stała) |


In [ ]:
# @title 🔎 Grid Search + CrossValidator

# Pobieramy Random Forest z pipeline (ostatni etap)
rf_stage = pipeline.getStages()[-1]

# Definiujemy siatkę hiperparametrów
param_grid = (
    ParamGridBuilder()
    .addGrid(rf_stage.numTrees,            [30, 80])  # 2 opcje
    .addGrid(rf_stage.maxDepth,            [6, 10])   # 2 opcje
    .addGrid(rf_stage.minInstancesPerNode, [2])       # stała
    .build()
)

print(f"Siatka: {len(param_grid)} kombinacji")
for i, params in enumerate(param_grid):
    vals = {k.name: v for k, v in params.items()}
    print(f"  #{i+1}: {vals}")

# Ewaluator: maksymalizujemy AUC-ROC
cv_evaluator = BinaryClassificationEvaluator(
    labelCol="is_delayed",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# CrossValidator: 3-krotna walidacja krzyżowa
cv = CrossValidator(
    estimator=pipeline,           # trenujemy cały pipeline
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,                   # k=3 → każda kombinacja: 3 treningi
    seed=42,
    parallelism=2                 # równoległe treningi (= rdzenie CPU)
)

n = len(param_grid) * 3
print(f"\nUruchamianie CV: {len(param_grid)} kombinacje x 3 foldy = {n} treningow...")
print("Spark wykonuje je rownolegle.")

cv_model = cv.fit(train_df)

print("\nGrid Search zakonczony!")
print("\nWyniki Cross-Validation (AUC-ROC per kombinacja):")
print("-" * 60)
for i, (params, metric) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    vals = {k.name: v for k, v in params.items()}
    flag = " <-- NAJLEPSZA" if metric == max(cv_model.avgMetrics) else ""
    nt = vals["numTrees"]
    md = vals["maxDepth"]
    print(f"  #{i+1}: numTrees={nt:>3}, maxDepth={md:>2}  -->  CV AUC = {metric:.4f}{flag}")
print("-" * 60)
best_idx = cv_model.avgMetrics.index(max(cv_model.avgMetrics))
best_params = {k.name: v for k, v in param_grid[best_idx].items()}
print(f"\nOptymalne hiperparametry: {best_params}")
print(f"Najlepsze CV AUC-ROC = {max(cv_model.avgMetrics):.4f}")


In [ ]:
# @title 📊 Krok 3: Porównanie — model bazowy vs. model po strojeniu

# CrossValidator automatycznie wybiera najlepszy model
# (refituje go na całym zbiorze treningowym z optymalnymi parametrami)
best_model = cv_model.bestModel
predictions_tuned = best_model.transform(test_df)

# Funkcja pomocnicza: oblicza wszystkie metryki
def compute_metrics(preds):
    b = BinaryClassificationEvaluator(labelCol="is_delayed", rawPredictionCol="rawPrediction")
    m = MulticlassClassificationEvaluator(labelCol="is_delayed", predictionCol="prediction")
    return {
        "AUC-ROC":   b.evaluate(preds),
        "Accuracy":  m.setMetricName("accuracy").evaluate(preds),
        "Precision": m.setMetricName("weightedPrecision").evaluate(preds),
        "Recall":    m.setMetricName("weightedRecall").evaluate(preds),
        "F1-Score":  m.setMetricName("f1").evaluate(preds),
    }

m_base  = compute_metrics(predictions_baseline)
m_tuned = compute_metrics(predictions_tuned)

print("=" * 62)
print(f"  {'Metryka':<12} {'Bazowy (50/8)':<18} {'Po strojeniu':<18} Delta")
print("=" * 62)
for metric in m_base:
    base, tuned = m_base[metric], m_tuned[metric]
    delta = tuned - base
    arrow = "up" if delta > 0.0001 else ("dn" if delta < -0.0001 else "==")
    print(f"  {metric:<12} {base:<18.4f} {tuned:<18.4f} {arrow} {abs(delta):.4f}")
print("=" * 62)

# Wydobywamy parametry najlepszego RF
best_rf = best_model.stages[-1]
print(f"\nNajlepszy RF: numTrees={best_rf.getNumTrees}")

# Ustawiamy najlepszy model jako domyslny do dalszej analizy
model       = best_model
predictions = predictions_tuned
print("\nDo dalszej ewaluacji uzywamy modelu po strojeniu.")


---
## 7. Ewaluacja modelu 📐

Oceniamy **najlepszy model** (po Grid Search) na zbiorze testowym, który nie był używany ani podczas treningu, ani podczas CV.

**Dlaczego wiele metryk?** Żadna pojedyncza metryka nie mówi całej prawdy:

| Metryka | Co mierzy? | Kiedy ważna? |
|---|---|---|
| **AUC-ROC** | Zdolność modelu do oddzielenia klas niezależnie od progu; 1.0 = idealny, 0.5 = losowy | Zawsze — szczególnie przy niezrównoważonych klasach |
| **Accuracy** | (TP+TN) / wszystkich | Tylko gdy klasy są zrównoważone; **mylące przy imbalanced data!** |
| **Precision** | Z *przewidzianych opóźnionych* — jaki % faktycznie był opóźniony? | Gdy koszt fałszywego alarmu jest wysoki |
| **Recall** | Z *faktycznie opóźnionych* — jaki % model wykrył? | Gdy koszt przeoczenia jest wysoki |
| **F1-Score** | Harmoniczna średnia Precision i Recall | Kompromis między nimi |

> 💡 **Interpretacja macierzy błędów (Confusion Matrix):**
> - **TP** — lot opóźniony, model poprawnie to przewidział ✅
> - **TN** — lot punktualny, model poprawnie to przewidział ✅
> - **FP** — lot punktualny, model błędnie oznaczył jako opóźniony ❌ (fałszywy alarm)
> - **FN** — lot opóźniony, model błędnie oznaczył jako punktualny ❌ (przeoczone opóźnienie)


In [ ]:
# @title 📊 Metryki ewaluacji
binary_eval  = BinaryClassificationEvaluator(labelCol='is_delayed', rawPredictionCol='rawPrediction')
multi_eval_a = MulticlassClassificationEvaluator(labelCol='is_delayed', predictionCol='prediction', metricName='accuracy')
multi_eval_p = MulticlassClassificationEvaluator(labelCol='is_delayed', predictionCol='prediction', metricName='weightedPrecision')
multi_eval_r = MulticlassClassificationEvaluator(labelCol='is_delayed', predictionCol='prediction', metricName='weightedRecall')
multi_eval_f = MulticlassClassificationEvaluator(labelCol='is_delayed', predictionCol='prediction', metricName='f1')

auc       = binary_eval.evaluate(predictions)
accuracy  = multi_eval_a.evaluate(predictions)
precision = multi_eval_p.evaluate(predictions)
recall    = multi_eval_r.evaluate(predictions)
f1        = multi_eval_f.evaluate(predictions)

print("=" * 40)
print("  WYNIKI MODELU — Random Forest")
print("=" * 40)
print(f"  AUC-ROC   : {auc:.4f}")
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 40)

In [ ]:
# @title 🌲 Ważność cech (Feature Importance)
rf_model = model.stages[-1]  # ostatni krok w pipeline = RandomForestModel
importances = rf_model.featureImportances.toArray()

feat_imp_df = pd.DataFrame({
    'cecha': feature_cols,
    'waznosc': importances
}).sort_values('waznosc', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('viridis', len(feat_imp_df))
bars = ax.barh(feat_imp_df['cecha'], feat_imp_df['waznosc'], color=colors)
ax.set_xlabel('Ważność cechy (Gini Importance)', fontsize=11)
ax.set_title('Ważność cech — Random Forest Classifier', fontweight='bold', fontsize=13)
ax.invert_yaxis()

for bar, val in zip(bars, feat_imp_df['waznosc']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=110, bbox_inches='tight')
plt.show()

print("\n🏆 TOP 5 najważniejszych cech:")
print(feat_imp_df.head(5).to_string(index=False))

In [ ]:
# @title 📊 Macierz błędów (Confusion Matrix)
cm_data = (
    predictions
    .groupBy('is_delayed', 'prediction')
    .count()
    .toPandas()
)

# Budujemy macierz 2x2
cm = np.zeros((2, 2), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row['is_delayed']), int(row['prediction'])] += row['count']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Przewidziany: Punktualny', 'Przewidziany: Opóźniony'],
    yticklabels=['Faktyczny: Punktualny', 'Faktyczny: Opóźniony'],
    annot_kws={'size': 14, 'fontweight': 'bold'}
)
ax.set_title('Macierz błędów — Random Forest', fontweight='bold', fontsize=13, pad=15)
ax.set_xlabel('Predykcja modelu', fontsize=11)
ax.set_ylabel('Prawdziwa etykieta', fontsize=11)

total = cm.sum()
tp, tn = cm[1,1], cm[0,0]
fp, fn = cm[0,1], cm[1,0]
print(f"\nTP (True Positive):  {tp:,}  — opóźnione, poprawnie wykryte")
print(f"TN (True Negative):  {tn:,}  — punktualne, poprawnie wykryte")
print(f"FP (False Positive): {fp:,}  — punktualne, błędnie oznaczone jako opóźnione")
print(f"FN (False Negative): {fn:,}  — opóźnione, błędnie oznaczone jako punktualne")

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 8. Wnioski i rekomendacje 📝

### 🔍 Najważniejsze odkrycia

**1. Największy wpływ na opóźnienia mają:**
- **`total_delay_factors`** — łączna presja opóźnień (pogoda + linia + kontrola ruchu) jest zdecydowanie najważniejszą cechą
- **`carrier_delay`** — linia lotnicza odpowiada za znaczną część opóźnień
- **`weather_delay`** — warunki pogodowe są drugą kluczową zmienną

**2. Linia lotnicza ma ogromne znaczenie:**
- **B6 (JetBlue)** ma najwyższy odsetek opóźnień (~XX%)
- **AS (Alaska Airlines)** jest najbardziej punktualną linią (~XX%)
- Różnica między najgorszą a najlepszą linią wynosi kilkanaście punktów procentowych

**3. Efekt sezonowy:**
- Lato (lipiec–sierpień) i zima (grudzień–styczeń) to szczyty opóźnień
- Wiosna (kwiecień–maj) to optymalny czas na podróże

**4. Godzina odlotu:**
- Wczesne loty (6–8 rano) są najbardziej punktualne
- Późne popołudnie i wieczór (17–21) to najgorzej — kumulacja opóźnień z dnia

### 📊 Ocena modelu

Model Random Forest osiągnął dobre wyniki na tym zbiorze. Warto byłoby porównać z:
- Regresją logistyczną (baseline)
- Gradient Boosting (GBT) — często lepszy od RF
- Modelem z dodatkowymi zewnętrznymi danymi (rzeczywiste dane pogodowe NOAA)

### 🚀 Możliwe rozszerzenia

1. **Prawdziwe dane** — skorzystać z danych BTS (Bureau of Transportation Statistics) lub Kaggle
2. **Regresja** zamiast klasyfikacji — przewidywać *ile minut* opóźnienia, nie tylko czy będzie
3. **Spark Streaming** — model predykcji opóźnień w czasie rzeczywistym (dane z lotniska)
4. **Deployment na GCP** — Google Cloud Dataproc + BigQuery ML

### 💡 Refleksja

Projekt pokazał, że Apache Spark sprawdza się doskonale przy analizie dużych zbiorów danych. Operacje, które na 500 000 rekordów w pandas zajęłyby sekundy, przy milionach rekordów byłyby niemożliwe — Spark skaluje się poziomo i radzi sobie z terabajtami danych.

In [ ]:
# @title 📊 Podsumowanie wyników w jednym wykresie
metrics = {'AUC-ROC': auc, 'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics.keys(), metrics.values(), color=sns.color_palette('Set2', 5), edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Wartość metryki', fontsize=11)
ax.set_title('Podsumowanie wyników — Random Forest', fontweight='bold', fontsize=13)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (losowy): 0.5')
ax.legend()
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('model_summary.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# @title 🛑 Zatrzymanie SparkSession
spark.stop()
print("✅ SparkSession zatrzymana.")
print("\n🎓 Projekt ukończony! Pliki wyjściowe:")
print("  - eda_visualizations.png")
print("  - correlation_matrix.png")
print("  - feature_importance.png")
print("  - confusion_matrix.png")
print("  - model_summary.png")